# Project 5 — Human Resource Analytics
**Level:** Intermediate  |  **Intern:** Sohan Nandi (CSN-OC-2026-474)  |  **Domain:** Data Analysis  |  **October 2026**

## Objective & Business Questions
Analyze employee attrition from the HR dataset (single-file version).

- What proportion of employees left?
- How does attrition vary by department and salary category?
- How are satisfaction, evaluation, project count, monthly hours, and tenure associated with attrition?
- How does attrition vary with promotion history or work-accident status?
- Which observed patterns may warrant further HR investigation?

**Scope note:** The internship task list describes four separate files (Employee, Salary, Attendance, Performance). The supplied file is a single table with satisfaction, evaluation, project count, hours, tenure, work accidents, attrition flag (`left`), promotion, department, and salary band. No separate attendance or detailed performance records are available. Analysis uses only available fields; limitations documented throughout.

In [1]:
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def resolve_repo_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").is_dir() and (parent / "projects").is_dir():
            return parent
        if (parent / "Dataset").is_dir():
            return parent
    return Path(__file__).resolve().parents[3]

REPO_ROOT = resolve_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
def fig_path(name):
    return str(OUT_DIR / name)

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUT_DIR)

# --- Load source data ---
SRC_FILE = DATASET_DIR / "HR_comma_sep.csv"
print("Source file:", SRC_FILE)
assert SRC_FILE.exists(), f"Source file missing: {SRC_FILE}"
df = pd.read_csv(SRC_FILE, encoding="latin1")
print("Loaded shape:", df.shape)


Repo root: C:\Users\sohan\OneDrive\Desktop\CyberNova
Dataset dir: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset
Output dir: outputs
Source file: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset\HR_comma_sep.csv


Loaded shape: (14999, 10)


In [2]:
print("dimensions:", df.shape)
print("columns:", list(df.columns))
print("dtypes:"); print(df.dtypes.to_string())
print("first 3 rows:"); print(df.head(3).to_string())

dimensions: (14999, 10)
columns: ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company', 'Work_accident', 'left', 'promotion_last_5years', 'Department', 'salary']
dtypes:
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
left                       int64
promotion_last_5years      int64
Department                   str
salary                       str
first 3 rows:
   satisfaction_level  last_evaluation  number_project  average_montly_hours  time_spend_company  Work_accident  left  promotion_last_5years Department  salary
0                0.38             0.53               2                   157                   3              0     1                      0      sales     low
1                0.80             0.86               5                   262                   6              0    

In [3]:
print("null counts:"); print(df.isnull().sum().to_string())
print("duplicate rows:", int(df.duplicated().sum()))
print("left (attrition) distribution:", df["left"].value_counts().to_dict())
print("attrition rate: %.2f%%" % df["left"].mean()*100)
print("Department:", df["Department"].value_counts().to_dict())
print("salary:", df["salary"].value_counts().to_dict())
print("Work_accident:", df["Work_accident"].value_counts().to_dict())
print("promotion_last_5years:", df["promotion_last_5years"].value_counts().to_dict())
for c in ["satisfaction_level","last_evaluation","number_project","average_montly_hours","time_spend_company"]:
    print(c, "range:", df[c].min(), "-", df[c].max())

null counts:
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
duplicate rows: 3008
left (attrition) distribution: {0: 11428, 1: 3571}
attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate: 0.24%attrition rate:

In [4]:
# Cleaning & validation
clean = df.copy()
# Validate ranges
assert clean["satisfaction_level"].between(0,1).all(), "satisfaction out of range"
assert clean["last_evaluation"].between(0,1).all(), "last_evaluation out of range"
assert clean["number_project"].between(1,10).all(), "number_project out of range"
assert clean["average_montly_hours"].between(80,400).all(), "hours out of range"
assert clean["time_spend_company"].between(1,20).all(), "tenure out of range"
print("cleaned rows (raw):", len(clean))

# Deduplicate (exact duplicates exist)
dup_count = int(clean.duplicated().sum())
print("exact duplicate rows:", dup_count)
# Primary analysis on deduplicated copy
dedup = clean.drop_duplicates().copy()
print("deduplicated rows:", len(dedup))
print("deduplicated attrition count:", int(dedup["left"].sum()))
print("deduplicated attrition rate: %.2f%%" % (dedup["left"].mean()*100))

# Compare key metrics before/after dedup
print("attrition rate (raw): %.2f%% | (dedup): %.2f%%" % (
    clean["left"].mean()*100, dedup["left"].mean()*100))
print("satisfaction mean (raw): %.3f | (dedup): %.3f" % (
    clean["satisfaction_level"].mean(), dedup["satisfaction_level"].mean()))

dedup.to_csv(OUT_DIR / "cleaned_hr_analytics.csv", index=False)
clean.to_csv(OUT_DIR / "cleaned_hr_analytics_raw.csv", index=False)
print("saved -> outputs/cleaned_hr_analytics.csv (dedup) and _raw.csv (with dupes)")

cleaned rows (raw): 14999
exact duplicate rows: 3008
deduplicated rows: 11991
deduplicated attrition count: 1991
deduplicated attrition rate: 16.60%
attrition rate (raw): 23.81% | (dedup): 16.60%
satisfaction mean (raw): 0.613 | (dedup): 0.630
saved -> outputs/cleaned_hr_analytics.csv (dedup) and _raw.csv (with dupes)


In [5]:
# Exploratory analysis on deduplicated data
d = dedup
print("--- attrition by department ---")
print(d.groupby("Department")["left"].agg(["count","mean"]).sort_values("mean", ascending=False).round(4).to_string())
print("--- attrition by salary ---")
print(d.groupby("salary")["left"].mean().sort_values(ascending=False).round(4).to_string())
print("--- attrition by work_accident ---")
print(d.groupby("Work_accident")["left"].mean().round(4).to_string())
print("--- attrition by promotion_last_5years ---")
print(d.groupby("promotion_last_5years")["left"].mean().round(4).to_string())
print("--- avg metrics by attrition ---")
print(d.groupby("left")[["satisfaction_level","last_evaluation","number_project",
                         "average_montly_hours","time_spend_company"]].mean().round(3).to_string())
print("--- attrition by project count ---")
print(d.groupby("number_project")["left"].mean().round(4).to_string())
print("--- attrition by tenure ---")
print(d.groupby("time_spend_company")["left"].mean().round(4).to_string())

--- attrition by department ---
             count    mean
Department                
hr             601  0.1880
accounting     621  0.1755
technical     2244  0.1738
support       1821  0.1713
sales         3239  0.1698
marketing      673  0.1664
IT             976  0.1619
product_mng    686  0.1603
RandD          694  0.1225
management     436  0.1193
--- attrition by salary ---
salary
low       0.2045
medium    0.1462
high      0.0485
--- attrition by work_accident ---
Work_accident
0    0.1860
1    0.0568
--- attrition by promotion_last_5years ---
promotion_last_5years
0    0.1682
1    0.0394
--- avg metrics by attrition ---
      satisfaction_level  last_evaluation  number_project  average_montly_hours  time_spend_company
left                                                                                               
0                  0.667            0.716           3.787               198.943               3.262
1                  0.440            0.722           3.883      

In [6]:
# Charts
import os
os.makedirs(OUT_DIR, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# attrition by department
d.groupby("Department")["left"].mean().sort_values(ascending=True).plot(
    kind="barh", ax=axes[0,0], color="#d62728")
axes[0,0].set_title("Attrition Rate by Department"); axes[0,0].set_xlabel("Attrition Rate")

# attrition by salary
d.groupby("salary")["left"].mean().sort_values(ascending=True).plot(
    kind="barh", ax=axes[0,1], color="#ff7f0e")
axes[0,1].set_title("Attrition Rate by Salary Category"); axes[0,1].set_xlabel("Attrition Rate")

# satisfaction distribution by left
for val, col, label in [(1, "#d62728", "Left"), (0, "#2ca02c", "Stayed")]:
    d.loc[d["left"]==val, "satisfaction_level"].plot(
        kind="hist", bins=20, ax=axes[0,2], alpha=0.5, color=col, label=label, edgecolor="white")
axes[0,2].set_title("Satisfaction by Attrition"); axes[0,2].set_xlabel("Satisfaction Level"); axes[0,2].legend()

# tenure (time_spend_company) by attrition
d.groupby("time_spend_company")["left"].mean().plot(
    kind="bar", ax=axes[1,0], color="#1f77b4")
axes[1,0].set_title("Attrition Rate by Tenure (years)"); axes[1,0].set_xlabel("Years at Company"); axes[1,0].set_ylabel("Attrition Rate")

# number_project by attrition
d.groupby("number_project")["left"].mean().plot(
    kind="bar", ax=axes[1,1], color="#9467bd")
axes[1,1].set_title("Attrition Rate by Project Count"); axes[1,1].set_xlabel("Number of Projects"); axes[1,1].set_ylabel("Attrition Rate")

# work accident & promotion
d.groupby("Work_accident")["left"].mean().plot(kind="bar", ax=axes[1,2], color="#8c564b", label="Work accident")
axes[1,2].set_title("Attrition: Work Accident & Promotion"); axes[1,2].set_xlabel("Category"); axes[1,2].set_ylabel("Attrition Rate")
# Add promotion as second bar
d.groupby("promotion_last_5years")["left"].mean().plot(kind="bar", ax=axes[1,2], color="#e377c2", label="Promotion", alpha=0.7)
axes[1,2].legend()

fig.tight_layout()
fig.savefig(fig_path("charts_hr.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved outputs/charts_hr.png")

saved outputs/charts_hr.png


In [7]:
print("KEY FINDINGS (hr analytics):")
print("1. Overall attrition: %.1f%% (%d of %d records, raw); %.1f%% (%d of %d dedup)" % (
    clean["left"].mean()*100, int(clean["left"].sum()), len(clean),
    dedup["left"].mean()*100, int(dedup["left"].sum()), len(dedup)))
print("2. Department highest attrition: HR %.1f%%, Accounting %.1f%%, Technical %.1f%%" % (
    d.groupby("Department")["left"].mean()["hr"]*100,
    d.groupby("Department")["left"].mean()["accounting"]*100,
    d.groupby("Department")["left"].mean()["technical"]*100))
print("3. Salary: low %.1f%%, medium %.1f%%, high %.1f%%" % (
    d.groupby("salary")["left"].mean()["low"]*100,
    d.groupby("salary")["left"].mean()["medium"]*100,
    d.groupby("salary")["left"].mean()["high"]*100))
print("4. Work accident: accident=0 attrition %.1f%%; accident=1 attrition %.1f%%" % (
    d.groupby("Work_accident")["left"].mean()[0]*100,
    d.groupby("Work_accident")["left"].mean()[1]*100))
print("5. Promotion: promoted=0 attrition %.1f%%; promoted=1 attrition %.1f%%" % (
    d.groupby("promotion_last_5years")["left"].mean()[0]*100,
    d.groupby("promotion_last_5years")["left"].mean()[1]*100))
print("6. Project count: 2 projects attrition %.1f%%; 7 projects attrition %.1f%% (U-shape)" % (
    d.groupby("number_project")["left"].mean().get(2,0)*100,
    d.groupby("number_project")["left"].mean().get(7,0)*100))
print("7. Satisfaction: left mean %.3f vs stayed %.3f" % (
    d.loc[d.left==1,"satisfaction_level"].mean(), d.loc[d.left==0,"satisfaction_level"].mean()))
print("8. Monthly hours: left mean %.1f vs stayed %.1f" % (
    d.loc[d.left==1,"average_montly_hours"].mean(), d.loc[d.left==0,"average_montly_hours"].mean()))
print("RECOMMENDATIONS:")
print("- Investigate HR/Accounting/Technical departments for culture/workload drivers.")
print("- Low-salary employees show highest attrition; review compensation bands.")
print("- Employees with 0 work accidents and 0 promotions have higher attrition; "
      "consider recognition and safety programs.")
print("- U-shaped project count risk: very few (2) and very many (7) projects both high attrition; "
      "review workload allocation.")
print("- Data limitation: single-file; no attendance logs or detailed performance reviews. "
      "No unique employee ID to verify dedup logic; 3008 exact duplicates removed for primary analysis.")

KEY FINDINGS (hr analytics):
1. Overall attrition: 23.8% (3571 of 14999 records, raw); 16.6% (1991 of 11991 dedup)
2. Department highest attrition: HR 18.8%, Accounting 17.6%, Technical 17.4%
3. Salary: low 20.5%, medium 14.6%, high 4.8%


4. Work accident: accident=0 attrition 18.6%; accident=1 attrition 5.7%
5. Promotion: promoted=0 attrition 16.8%; promoted=1 attrition 3.9%
6. Project count: 2 projects attrition 54.2%; 7 projects attrition 100.0% (U-shape)
7. Satisfaction: left mean 0.440 vs stayed 0.667
8. Monthly hours: left mean 208.2 vs stayed 198.9
RECOMMENDATIONS:
- Investigate HR/Accounting/Technical departments for culture/workload drivers.
- Low-salary employees show highest attrition; review compensation bands.
- Employees with 0 work accidents and 0 promotions have higher attrition; consider recognition and safety programs.
- U-shaped project count risk: very few (2) and very many (7) projects both high attrition; review workload allocation.
- Data limitation: single-file; no attendance logs or detailed performance reviews. No unique employee ID to verify dedup logic; 3008 exact duplicates removed for primary analysis.


In [8]:
print("LIMITATIONS:")
print("- Single CSV; no separate attendance, salary history, or performance tables as task list described.")
print("- Exact duplicates removed (3008 rows); no employee ID to confirm they are true duplicates vs multiple records per person.")
print("- Observational: associations shown, not causal claims (e.g., low satisfaction does not prove it caused leaving).")
print("- Sensitive employee data: avoid publishing individual-level records in screenshots.")
print("- Results on deduped data differ from raw (attrition rate 16.6% vs 23.8%); both reported for transparency.")
print("CONCLUSION: Attrition is highest in HR, Accounting, and Technical departments, and among low-salary employees. "
      "Work accident and promotion history show protective associations. "
      "Project count and monthly hours exhibit U-shaped risk. Further HR investigation recommended.")

LIMITATIONS:
- Single CSV; no separate attendance, salary history, or performance tables as task list described.
- Exact duplicates removed (3008 rows); no employee ID to confirm they are true duplicates vs multiple records per person.
- Observational: associations shown, not causal claims (e.g., low satisfaction does not prove it caused leaving).
- Sensitive employee data: avoid publishing individual-level records in screenshots.
- Results on deduped data differ from raw (attrition rate 16.6% vs 23.8%); both reported for transparency.
CONCLUSION: Attrition is highest in HR, Accounting, and Technical departments, and among low-salary employees. Work accident and promotion history show protective associations. Project count and monthly hours exhibit U-shaped risk. Further HR investigation recommended.
